In [1]:
import pandas as pd

# Load the three dataframes
df_artists = pd.read_csv('/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_artists.csv')
df_songs = pd.read_csv('/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_songs.csv')
df_albums = pd.read_csv('/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_albums.csv')

# Display basic info
print("df_artists:")
print(f"  Shape: {df_artists.shape}")
print(f"  Columns: {list(df_artists.columns)}")

print("\ndf_songs:")
print(f"  Shape: {df_songs.shape}")
print(f"  Columns: {list(df_songs.columns)}")

print("\ndf_albums:")
print(f"  Shape: {df_albums.shape}")
print(f"  Columns: {list(df_albums.columns)}")


df_artists:
  Shape: (14226, 55)
  Columns: ['performer_normalized', 'performer', 'first_song_year', 'last_song_year', 'first_album_year', 'last_album_year', 'years_active_on_charts', '#_of_unique_years_active', 'total_charting_songs', 'total_charting_albums', '#1_hit_song_count', '#1_hit_album_count', 'top_10_song_count', 'top_10_album_count', 'top_20_song_count', 'top_20_album_count', 'top_50_song_count', 'top_50_album_count', 'highest_charting_song_name', 'highest_charting_song_position', 'first_charting_song_name', 'first_charting_song_position', 'first_charting_song_duration', 'total_charting_songs_duration_weeks', 'highest_charting_album_name', 'highest_charting_album_position', 'first_charting_album_name', 'first_charting_album_position', 'first_charting_album_duration', 'total_charting_albums_duration_weeks', 'tags', 'genre_tags_musicbrainz', 'first_year_on_chart_songs', 'first_year_top_20_songs', 'first_year_top_10_songs', 'first_year_num1_songs', 'first_year_on_chart_albums',

In [2]:
"""
Add MusicBrainz MBID (UUID) to df_artists
==========================================

This code queries the local MusicBrainz PostgreSQL database to retrieve the 
official MusicBrainz Identifier (MBID) for each artist in df_artists.

The MBID is a UUID (e.g., '20244d07-534f-4eff-b4d4-930878889970') stored in 
the artist.gid column of the MusicBrainz database. Unlike the internal integer 
ID (artist.id), the MBID is:
  - Stable across all MusicBrainz instances
  - The official public identifier for external references
  - Used by the MusicBrainz API and other music services

Matching method:
  - Case-insensitive exact string match on artist name
  - Billboard's 'performer_normalized' field → MusicBrainz 'artist.name' field
  - If multiple artists have the same name, takes the first match

Output:
  - Adds a new column 'musicbrainz_mbid' to df_artists
  - Contains UUID strings for matched artists, NaN for unmatched
"""

import pandas as pd
import psycopg2
from tqdm import tqdm

# Database connection parameters
DB_PARAMS = {
    'dbname': 'musicbrainz_db',
    'user': 'musicbrainz',
    'password': 'musicbrainz',
    'host': 'localhost',
    'port': 5432
}

print("Step 1: Collecting unique artist names from df_artists...")
all_artist_names = df_artists['performer_normalized'].dropna().unique().tolist()
print(f"  Found {len(all_artist_names):,} unique artist names")

# Step 2: Query MusicBrainz database for MBIDs
print("\nStep 2: Querying MusicBrainz for artist MBIDs...")

artist_name_to_mbid = {}
batch_size = 1000

conn = psycopg2.connect(**DB_PARAMS)
conn.autocommit = True

num_batches = (len(all_artist_names) + batch_size - 1) // batch_size

with conn.cursor() as cur:
    for i in tqdm(range(num_batches), desc="Processing batches"):
        batch_names = all_artist_names[i * batch_size:(i + 1) * batch_size]
        
        # Query for artist gid (MBID)
        query = """
            SELECT DISTINCT ON (LOWER(input_name)) 
                input_name,
                artist.gid,
                artist.name
            FROM unnest(%s::text[]) AS input_name
            LEFT JOIN artist ON LOWER(artist.name) = LOWER(input_name)
            WHERE artist.gid IS NOT NULL
        """
        
        cur.execute(query, (batch_names,))
        results = cur.fetchall()
        
        # Store mappings
        for input_name, artist_gid, matched_name in results:
            artist_name_to_mbid[input_name] = str(artist_gid)

conn.close()

print(f"  Successfully matched: {len(artist_name_to_mbid):,} artists")
print(f"  Not found: {len(all_artist_names) - len(artist_name_to_mbid):,} artists")

# Step 3: Add musicbrainz_mbid column to df_artists
print("\nStep 3: Adding musicbrainz_mbid column to df_artists...")

df_artists['musicbrainz_mbid'] = df_artists['performer_normalized'].map(artist_name_to_mbid)

matched = df_artists['musicbrainz_mbid'].notna().sum()
total = len(df_artists)
print(f"  Matched: {matched:,}/{total:,} rows ({matched/total*100:.1f}%)")

# Step 4: Display sample results
print("\nStep 4: Sample results:")
print("="*80)
sample = df_artists[df_artists['musicbrainz_mbid'].notna()][['performer_normalized', 'musicbrainz_mbid']].head(10)
print(sample.to_string(index=False))

print("\n" + "="*80)
print("COMPLETE: df_artists now has 'musicbrainz_mbid' column")
print("="*80)
print("\nTo save: df_artists.to_csv('df_artists.csv', index=False)")


Step 1: Collecting unique artist names from df_artists...
  Found 14,226 unique artist names

Step 2: Querying MusicBrainz for artist MBIDs...


Processing batches: 100%|██████████| 15/15 [00:25<00:00,  1.71s/it]

  Successfully matched: 11,564 artists
  Not found: 2,662 artists

Step 3: Adding musicbrainz_mbid column to df_artists...
  Matched: 11,564/14,226 rows (81.3%)

Step 4: Sample results:
performer_normalized                     musicbrainz_mbid
                $not d993169f-2033-4810-bae8-564d4aab89cd
         $uicideboy$ 0d23ca32-6136-4522-8cc2-1b624d0ea2d1
              *nsync 603ba565-3967-4be1-931e-9cb945394e86
           070 shake 741dbc8e-1cd5-4a66-b2b8-43bbad87ae12
      1 of the girls c653c820-39ad-4700-9269-a38b61f5f6c6
            10 years b18bc9c4-6f22-4f1b-a918-e9c86a39fe7a
      10,000 maniacs b9a06530-1241-4162-836f-7b8e79deaa58
            100 gecs d6d45dda-2377-4faa-947b-e071efa085c0
         100 strings f8015955-3780-48d2-b030-e8098e6b2ad2
                10cc f37c537b-3557-4031-bfd6-ab63ced32854

COMPLETE: df_artists now has 'musicbrainz_mbid' column

To save: df_artists.to_csv('df_artists.csv', index=False)


In [3]:
# Save df_artists with the new musicbrainz_mbid column
output_path = '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_artists.csv'

print(f"Saving df_artists to {output_path}...")
df_artists.to_csv(output_path, index=False)
print("✓ Saved successfully")

print(f"\nFinal df_artists shape: {df_artists.shape}")
print(f"Columns: {len(df_artists.columns)}")


Saving df_artists to /Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_artists.csv...
✓ Saved successfully

Final df_artists shape: (14226, 56)
Columns: 56


In [4]:
"""
Validation Check: Verify MBID Assignment Accuracy
==================================================

This code validates that the musicbrainz_mbid column was correctly assigned by:
  1. Sampling 100 random rows from df_artists (with non-null MBIDs)
  2. For each MBID, querying MusicBrainz to get the official artist name
  3. Comparing the Billboard artist name with the MusicBrainz artist name
  4. Reporting any mismatches

This catches issues like:
  - Wrong MBID assigned due to name ambiguity
  - Database query errors
  - Name matching problems
"""

import pandas as pd
import psycopg2

# Database connection
DB_PARAMS = {
    'dbname': 'musicbrainz_db',
    'user': 'musicbrainz',
    'password': 'musicbrainz',
    'host': 'localhost',
    'port': 5432
}

print("Validation Check: Sampling 100 random artists with MBIDs...")
print("="*80)

# Sample 100 random rows that have an MBID
sample_df = df_artists[df_artists['musicbrainz_mbid'].notna()].sample(n=min(100, df_artists['musicbrainz_mbid'].notna().sum()), random_state=42)

print(f"Sampled {len(sample_df)} rows for validation\n")

# Connect to database
conn = psycopg2.connect(**DB_PARAMS)
conn.autocommit = True

mismatches = []
matches = []

with conn.cursor() as cur:
    for idx, row in sample_df.iterrows():
        billboard_name = row['performer_normalized']
        mbid = row['musicbrainz_mbid']
        
        # Query MusicBrainz for this MBID
        query = """
            SELECT name, gid
            FROM artist
            WHERE gid = %s::uuid
        """
        
        cur.execute(query, (mbid,))
        result = cur.fetchone()
        
        if result:
            mb_name, mb_gid = result
            
            # Check if names match (case-insensitive)
            if billboard_name.lower() == mb_name.lower():
                matches.append({
                    'billboard_name': billboard_name,
                    'musicbrainz_name': mb_name,
                    'mbid': mbid,
                    'status': 'MATCH ✓'
                })
            else:
                mismatches.append({
                    'billboard_name': billboard_name,
                    'musicbrainz_name': mb_name,
                    'mbid': mbid,
                    'status': 'MISMATCH ✗'
                })
        else:
            mismatches.append({
                'billboard_name': billboard_name,
                'musicbrainz_name': 'NOT FOUND IN DB',
                'mbid': mbid,
                'status': 'ERROR ✗'
            })

conn.close()

# Report results
print(f"Validation Results:")
print(f"  Matches: {len(matches)}/{len(sample_df)} ({len(matches)/len(sample_df)*100:.1f}%)")
print(f"  Mismatches: {len(mismatches)}/{len(sample_df)} ({len(mismatches)/len(sample_df)*100:.1f}%)")

if mismatches:
    print(f"\n⚠️  Found {len(mismatches)} mismatches:")
    print("="*80)
    mismatch_df = pd.DataFrame(mismatches)
    print(mismatch_df.to_string(index=False))
else:
    print("\n✓ All sampled rows match correctly!")

# Show a few example matches
print(f"\nSample of correct matches (first 10):")
print("="*80)
if matches:
    match_df = pd.DataFrame(matches[:10])
    print(match_df[['billboard_name', 'musicbrainz_name', 'mbid']].to_string(index=False))

print("\n" + "="*80)
print("Validation check complete")
print("="*80)


Validation Check: Sampling 100 random artists with MBIDs...
Sampled 100 rows for validation

Validation Results:
  Matches: 100/100 (100.0%)
  Mismatches: 0/100 (0.0%)

✓ All sampled rows match correctly!

Sample of correct matches (first 10):
   billboard_name  musicbrainz_name                                 mbid
      nick cannon       Nick Cannon d6215935-98cf-464d-93bb-cc5fd58518bd
       jann arden        Jann Arden 31930c85-3e0e-488e-8407-46a0d73fa816
     terry dexter      Terry Dexter 88a79963-e4fc-4986-9ba3-2307c8dfbbeb
   the chordettes    The Chordettes 997117eb-5a4c-4a37-99f9-f2515e2d9739
      buju banton       Buju Banton 9f525d0b-3911-4c83-b0d1-e90aa1fd2d14
       dem jointz        Dem Jointz b228cdc8-0f48-47b5-b812-62fdc416bad3
        mike post         Mike Post 231d9912-9fcf-4aaf-8817-12d737ff858e
           gungor            Gungor f68ad842-13b9-4302-8eeb-ade8af70ce96
michelle williams Michelle Williams cb89b94b-1ee1-4b34-aaa6-23453fd254e1
       rufus blaq        R

In [8]:
"""
Add MusicBrainz Recording MBID to df_songs (CORRECTED)
=======================================================

Uses performer_normalized instead of original_performer to match MusicBrainz 
artist names (which we already successfully matched for df_artists).

Note: Each song may have multiple rows (one per performer in a collaboration).
This code matches any performer on the recording.
"""

import pandas as pd
import psycopg2
from tqdm import tqdm

DB_PARAMS = {
    'dbname': 'musicbrainz_db',
    'user': 'musicbrainz',
    'password': 'musicbrainz',
    'host': 'localhost',
    'port': 5432
}

print("Step 1: Collecting unique (performer, song) pairs from df_songs...")

# Use performer_normalized instead of original_performer
df_songs_unique = df_songs[['performer_normalized', 'title']].dropna().drop_duplicates()
print(f"  Found {len(df_songs_unique):,} unique (performer, song) pairs")

# Create list of tuples
song_pairs = list(df_songs_unique.itertuples(index=False, name=None))

print("\nStep 2: Querying MusicBrainz for recording MBIDs...")

song_to_mbid = {}
batch_size = 500

conn = psycopg2.connect(**DB_PARAMS)
conn.autocommit = True

num_batches = (len(song_pairs) + batch_size - 1) // batch_size

with conn.cursor() as cur:
    for i in tqdm(range(num_batches), desc="Processing batches"):
        batch_pairs = song_pairs[i * batch_size:(i + 1) * batch_size]
        
        artist_names = [pair[0] for pair in batch_pairs]
        song_titles = [pair[1] for pair in batch_pairs]
        
        query = """
            WITH input_data AS (
                SELECT 
                    unnest(%s::text[]) AS input_artist,
                    unnest(%s::text[]) AS input_title
            )
            SELECT DISTINCT ON (LOWER(input_data.input_artist), LOWER(input_data.input_title))
                input_data.input_artist,
                input_data.input_title,
                recording.gid,
                recording.name,
                artist.name AS artist_name
            FROM input_data
            JOIN recording ON LOWER(recording.name) = LOWER(input_data.input_title)
            JOIN artist_credit ON recording.artist_credit = artist_credit.id
            JOIN artist_credit_name ON artist_credit.id = artist_credit_name.artist_credit
            JOIN artist ON artist_credit_name.artist = artist.id
            WHERE LOWER(artist.name) = LOWER(input_data.input_artist)
        """
        
        cur.execute(query, (artist_names, song_titles))
        results = cur.fetchall()
        
        for input_artist, input_title, rec_gid, rec_name, mb_artist in results:
            song_to_mbid[(input_artist, input_title)] = str(rec_gid)

conn.close()

print(f"  Successfully matched: {len(song_to_mbid):,} songs")
print(f"  Not found: {len(song_pairs) - len(song_to_mbid):,} songs")

# Step 3: Add musicbrainz_recording_mbid column
print("\nStep 3: Adding musicbrainz_recording_mbid column to df_songs...")

def get_recording_mbid(row):
    if pd.notna(row['performer_normalized']) and pd.notna(row['title']):
        return song_to_mbid.get((row['performer_normalized'], row['title']))
    return None

df_songs['musicbrainz_recording_mbid'] = df_songs.apply(get_recording_mbid, axis=1)

matched = df_songs['musicbrainz_recording_mbid'].notna().sum()
total = len(df_songs)
print(f"  Matched: {matched:,}/{total:,} rows ({matched/total*100:.1f}%)")

# Step 4: Sample results
print("\nStep 4: Sample results:")
print("="*80)
sample = df_songs[df_songs['musicbrainz_recording_mbid'].notna()][
    ['performer_normalized', 'title', 'musicbrainz_recording_mbid']
].head(10)
print(sample.to_string(index=False))

print("\n" + "="*80)
print("COMPLETE: df_songs now has 'musicbrainz_recording_mbid' column")
print("="*80)


Step 1: Collecting unique (performer, song) pairs from df_songs...
  Found 4,911 unique (performer, song) pairs

Step 2: Querying MusicBrainz for recording MBIDs...


Processing batches: 100%|██████████| 10/10 [03:16<00:00, 19.65s/it]

  Successfully matched: 3,929 songs
  Not found: 982 songs

Step 3: Adding musicbrainz_recording_mbid column to df_songs...
  Matched: 3,981/5,099 rows (78.1%)

Step 4: Sample results:
performer_normalized                       title           musicbrainz_recording_mbid
       tommy edwards        It's All In The Game 11363e82-cba9-43e1-b7be-c0d600fc19aa
       conway twitty      It's Only Make Believe e92fc140-2030-43eb-ae5a-9f6d40033c62
        the elegants                 Little Star 99bdbbd3-eb9e-4d89-916b-bdbf654793cd
        ricky nelson            Poor Little Fool 100b7f30-1519-427a-adc2-2b0d6a737db1
        the platters     Smoke Gets In Your Eyes 1287244f-8411-4180-8f16-22bc7ace7fcc
         lloyd price                 Stagger Lee a6d35950-015e-42ed-a06b-00b1db08c7d2
       the chipmunks           The Chipmunk Song 3e4d4e58-8772-4f3c-9e14-9c0abe03c945
       david seville           The Chipmunk Song 91d763b7-3e00-4059-90b3-e1023d28d27f
     the teddy bears To Know Him, Is To L

In [9]:
"""
Add MusicBrainz Release Group MBID to df_albums
================================================

This code queries the MusicBrainz database to retrieve the MBID for each album 
in df_albums.

For albums, we match on:
  - Album title (df_albums['title'] → MusicBrainz release_group.name)
  - Artist name (df_albums['performer_normalized'] → MusicBrainz artist.name)

Uses release_group (not release) because:
  - release_group = the abstract album concept
  - release = specific editions (US, UK, remaster, etc.)
  - release_group is better for analysis (groups all versions together)

The MBID comes from release_group.gid

Output:
  - Adds a new column 'musicbrainz_release_group_mbid' to df_albums
"""

import pandas as pd
import psycopg2
from tqdm import tqdm

DB_PARAMS = {
    'dbname': 'musicbrainz_db',
    'user': 'musicbrainz',
    'password': 'musicbrainz',
    'host': 'localhost',
    'port': 5432
}

print("Step 1: Collecting unique (performer, album) pairs from df_albums...")

# Use performer_normalized (learned from songs debugging)
df_albums_unique = df_albums[['performer_normalized', 'title']].dropna().drop_duplicates()
print(f"  Found {len(df_albums_unique):,} unique (performer, album) pairs")

# Create list of tuples
album_pairs = list(df_albums_unique.itertuples(index=False, name=None))

print("\nStep 2: Querying MusicBrainz for release group MBIDs...")

album_to_mbid = {}
batch_size = 500

conn = psycopg2.connect(**DB_PARAMS)
conn.autocommit = True

num_batches = (len(album_pairs) + batch_size - 1) // batch_size

with conn.cursor() as cur:
    for i in tqdm(range(num_batches), desc="Processing batches"):
        batch_pairs = album_pairs[i * batch_size:(i + 1) * batch_size]
        
        artist_names = [pair[0] for pair in batch_pairs]
        album_titles = [pair[1] for pair in batch_pairs]
        
        # Query release_group table (albums)
        query = """
            WITH input_data AS (
                SELECT 
                    unnest(%s::text[]) AS input_artist,
                    unnest(%s::text[]) AS input_title
            )
            SELECT DISTINCT ON (LOWER(input_data.input_artist), LOWER(input_data.input_title))
                input_data.input_artist,
                input_data.input_title,
                release_group.gid,
                release_group.name,
                artist.name AS artist_name
            FROM input_data
            JOIN release_group ON LOWER(release_group.name) = LOWER(input_data.input_title)
            JOIN artist_credit ON release_group.artist_credit = artist_credit.id
            JOIN artist_credit_name ON artist_credit.id = artist_credit_name.artist_credit
            JOIN artist ON artist_credit_name.artist = artist.id
            WHERE LOWER(artist.name) = LOWER(input_data.input_artist)
        """
        
        cur.execute(query, (artist_names, album_titles))
        results = cur.fetchall()
        
        for input_artist, input_title, rg_gid, rg_name, mb_artist in results:
            album_to_mbid[(input_artist, input_title)] = str(rg_gid)

conn.close()

print(f"  Successfully matched: {len(album_to_mbid):,} albums")
print(f"  Not found: {len(album_pairs) - len(album_to_mbid):,} albums")

# Step 3: Add musicbrainz_release_group_mbid column
print("\nStep 3: Adding musicbrainz_release_group_mbid column to df_albums...")

def get_release_group_mbid(row):
    if pd.notna(row['performer_normalized']) and pd.notna(row['title']):
        return album_to_mbid.get((row['performer_normalized'], row['title']))
    return None

df_albums['musicbrainz_release_group_mbid'] = df_albums.apply(get_release_group_mbid, axis=1)

matched = df_albums['musicbrainz_release_group_mbid'].notna().sum()
total = len(df_albums)
print(f"  Matched: {matched:,}/{total:,} rows ({matched/total*100:.1f}%)")

# Step 4: Sample results
print("\nStep 4: Sample results:")
print("="*80)
sample = df_albums[df_albums['musicbrainz_release_group_mbid'].notna()][
    ['performer_normalized', 'title', 'musicbrainz_release_group_mbid']
].head(10)
print(sample.to_string(index=False))

print("\n" + "="*80)
print("COMPLETE: df_albums now has 'musicbrainz_release_group_mbid' column")
print("="*80)
print("\nTo save: df_albums.to_csv('df_albums.csv', index=False)")


Step 1: Collecting unique (performer, album) pairs from df_albums...
  Found 4,473 unique (performer, album) pairs

Step 2: Querying MusicBrainz for release group MBIDs...


Processing batches: 100%|██████████| 9/9 [00:38<00:00,  4.29s/it]

  Successfully matched: 3,161 albums
  Not found: 1,312 albums

Step 3: Adding musicbrainz_release_group_mbid column to df_albums...
  Matched: 3,164/4,569 rows (69.2%)

Step 4: Sample results:
           performer_normalized                                 title       musicbrainz_release_group_mbid
herb alpert & the tijuana brass                      !!Going Places!! 3f762d15-18ba-3883-a7b4-73a0c9f39f46
                    the monkees                          Headquarters 60ea3167-8594-3d42-9904-e48efaea09b3
          the mamas & the papas If You Can Believe Your Eyes And Ears 6399b0c8-a1a0-3674-92b5-585a574fdf20
                    the monkees                   More Of The Monkees 60d3909e-2ef6-36fa-bf36-269c5ff82566
                  bobbie gentry                     Ode To Billie Joe 7b747806-2720-318f-b984-3d72d04a6504
                    the beatles                              Revolver 72d15666-99a7-321e-b1f3-a3f8c09dff9f
                    the monkees                          

In [10]:
# Save df_songs and df_albums with the new MusicBrainz MBID columns

print("Saving dataframes...")
print("="*80)

# Save df_songs
songs_path = '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_songs.csv'
print(f"\nSaving df_songs to {songs_path}...")
df_songs.to_csv(songs_path, index=False)
print(f"✓ Saved df_songs")
print(f"  Shape: {df_songs.shape}")
print(f"  Columns: {len(df_songs.columns)}")

# Save df_albums
albums_path = '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_albums.csv'
print(f"\nSaving df_albums to {albums_path}...")
df_albums.to_csv(albums_path, index=False)
print(f"✓ Saved df_albums")
print(f"  Shape: {df_albums.shape}")
print(f"  Columns: {len(df_albums.columns)}")

print("\n" + "="*80)
print("COMPLETE: Both dataframes saved successfully")
print("="*80)


Saving dataframes...

Saving df_songs to /Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_songs.csv...
✓ Saved df_songs
  Shape: (5099, 18)
  Columns: 18

Saving df_albums to /Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_albums.csv...
✓ Saved df_albums
  Shape: (4569, 18)
  Columns: 18

COMPLETE: Both dataframes saved successfully
